# Physical Axicon Beam Study

Stage C notebook for the physical refractive axicon path with SLM2 conjugate flattening.

In [1]:
from dataclasses import replace
from pathlib import Path
import pandas as pd
import bessel_twin_core as bt
from vbb_study import vbb_regime, vbb_train_viz
base = replace(bt.default_config('fast'), generation_method='physical')
project_dir = Path.cwd() if Path.cwd().name == 'Publication_Study' else Path.cwd()/'Publication_Study'
out_fig = project_dir/'outputs/figures/stage_c'
out_csv = project_dir/'outputs/csv/stage_c'
out_fig.mkdir(parents=True, exist_ok=True); out_csv.mkdir(parents=True, exist_ok=True)

In [2]:
rows = []
for regime in ('general', 'limits'):
    cfg = vbb_regime.config_for_regime(base, regime)
    ideal_cfg = replace(cfg, physical_axicon=replace(cfg.physical_axicon, slm2_stroke_levels=None, slm2_conjugate_mode='full'))
    lab_cfg = replace(cfg, physical_axicon=replace(cfg.physical_axicon, slm2_stroke_levels=256, slm2_conjugate_mode='full'))
    for label, run_cfg in [('ideal', ideal_cfg), ('lab', lab_cfg)]:
        result = bt.run_case(run_cfg, preset='fast', path='ideal', case_id=f'{regime}_physical_{label}')
        m = result['metrics']; meta = result['axicon_metadata']
        rows.append({'regime': regime, 'path': label, 'zone_um': m['bessel_zone_um'], 'feature_um': m['feature_diameter_um'], 'peak_fluence_J_cm2': m['peak_fluence_J_cm2'], 'contrast': m['side_to_core_peak_ratio'], 'throughput': m['first_order_selected_fraction'], 'k_r_m_inv': result['axicon_result'].k_r, 'slm2_before_rad': meta['slm2_residual_phase_rms_before_rad'], 'slm2_after_rad': meta['slm2_residual_phase_rms_after_rad'], 'valid': result['validity_report']['valid']})
summary = pd.DataFrame(rows)
summary.to_csv(out_csv/'NB_physical_summary.csv', index=False)
summary

,regime,path,zone_um,feature_um,peak_fluence_J_cm2,contrast,throughput,k_r_m_inv,slm2_before_rad,slm2_after_rad,valid
0,general,ideal,113.947543,4.775000,14.674487,0.602715,0.950288,1.603333e+06,1.813817,0.000000,True
1,general,lab,113.937100,4.775000,14.381208,0.539002,0.950288,1.603333e+06,1.813817,0.007127,True
2,limits,ideal,103.835818,3.267105,14.807410,0.104699,0.849770,2.405000e+06,1.813146,0.000000,True
3,limits,lab,103.829493,3.267105,14.810231,0.104739,0.849770,2.405000e+06,1.813146,0.007086,True


In [3]:
vbb_train_viz.plot_train_visualiser(base, method='physical', output_dir=out_fig)
vbb_train_viz.plot_sampling_qa(base, output_dir=out_fig)

WindowsPath('C:/PhD/Code/Publication_Study/outputs/figures/stage_c/stage_c_sampling_qa_limits.png')